# Colab setup (run top to bottom)
1. Runtime > Change runtime type > T4 GPU.
2. Change YOUR_NAME below, then run each cell in order.
3. You will be asked once to paste your Kaggle API token (never saved in this notebook).
4. Data is downloaded from Kaggle into the Colab VM (not git). It is wiped when the session ends; just re-run.

In [1]:
import os
from getpass import getpass

os.environ['RUN_USER'] = 'Kevin Yang'

!rm -rf /content/repo
!git clone -q -b Kevin_Experiment https://github.com/JerNB/daml-indoor67.git /content/repo

%cd /content/repo

!pip -q install timm kaggle

import torch
print('GPU available:', torch.cuda.is_available())

!git branch --show-current

/content/repo
GPU available: True
Kevin_Experiment


In [2]:
# 2) data: download from Kaggle (your account must have joined the competition and accepted the rules)
os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle API token: ')
!mkdir -p /content/dl /content/data
!kaggle competitions download -c daml-aitp-fall-26-challenge -p /content/dl
!cd /content/dl && unzip -q -o '*.zip' -d /content/data && ls /content/data

Kaggle API token: ··········
100% 272M/272M [00:15<00:00, 18.4MB/s]

classes.csv  images  sample_submission.csv  test.csv  train.csv


In [3]:
# 3) locate the folder that holds train.csv (handles an extra nesting level) and verify layout
from pathlib import Path
hits = list(Path('/content/data').rglob('train.csv'))
assert hits, 'train.csv not found - check the download cell output'
root = hits[0].parent
os.environ['DATA_ROOT'] = str(root)
for need in ['classes.csv', 'sample_submission.csv', 'images/train', 'images/test']:
    print(need, 'OK' if (root / need).exists() else 'MISSING')
print('DATA_ROOT =', root)

classes.csv OK
sample_submission.csv OK
images/train OK
images/test OK
DATA_ROOT = /content/data


In [4]:
# 4) smoke test: baseline end to end (a few minutes on a GPU)
!python src/extract_features.py --model resnet50.a1_in1k
!python src/linear_probe.py --model resnet50.a1_in1k --name baseline
!python src/summarize.py


model.safetensors: downloading bytes:  88% 90.6M/102M [00:02<00:00, 74.2MB/s, 7.69MB/s  ]
model.safetensors: reconstructing file:  78% 79.8M/102M [00:02<00:00, 34.0MB/s]
model.safetensors: downloading bytes: 100% 96.2M/96.2M [00:02<00:00, 35.9MB/s, 8.67MB/s  ]
model.safetensors: reconstructing file: 100% 102M/102M [00:02<00:00, 38.2MB/s, 9.56MB/s  ]
train: 100% 84/84 [00:29<00:00,  2.85it/s]
test: 100% 21/21 [00:06<00:00,  3.14it/s]
val_acc (fold 0) = 0.7711
saved results/runs/20261126-030928_Kevin Yang_baseline.json
    name       user            model   C  fold  seed  val_acc            time
baseline Kevin Yang resnet50.a1_in1k 1.0     0    42 0.771107 20261126-030928


In [5]:
!git fetch origin
!git branch -a

!ls src

!ls notebooks

* Kevin_Experiment
  remotes/origin/Cheney-Experiment
  remotes/origin/HEAD -> origin/main
  remotes/origin/Jerry-Experiment
  remotes/origin/Kevin_Experiment
  remotes/origin/Lydia_Experiments
  remotes/origin/Sylvia-Experiment
  remotes/origin/main
common.py	     linear_probe.py	 __pycache__  summarize.py
extract_features.py  make_submission.py  split.py
colab_setup.ipynb


In [6]:
!git ls-tree -r --name-only origin/Jerry-Experiment -- src notebooks

notebooks/colab_setup.ipynb
src/common.py
src/extract_features.py
src/finetune.py
src/linear_probe.py
src/make_submission.py
src/split.py
src/summarize.py


In [7]:
!git show origin/Jerry-Experiment:src/finetune.py

"""Fine-tune an ImageNet-pretrained backbone end to end (all layers unfrozen).

Difference from the baseline: the baseline froze the network and only trained a
logistic regression on its features. Here the whole network keeps learning on
Indoor67, so the features themselves adapt to indoor scenes.

Examples:
  # model selection: train on folds 1-4, validate on fold 0
  python src/finetune.py --model convnext_base.fb_in1k --epochs 10 --name ft_convnext
  # final run: train on ALL labeled images, then write a Kaggle submission
  python src/finetune.py --model convnext_base.fb_in1k --epochs 10 --fold -1 --name ft_convnext_full
"""
import argparse, json, math, time
from pathlib import Path

import pandas as pd
import torch
import timm
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

from common import data_root, set_seed, who, SEED

# -------------------------------------------------------------------------

In [9]:
!git restore --source=origin/Jerry-Experiment --worktree src/finetune.py

!git branch --show-current

!ls src

!git diff HEAD origin/Jerry-Experiment -- src/common.py splits/folds.csv

!python src/finetune.py --help

Kevin_Experiment
common.py	     finetune.py      make_submission.py  split.py
extract_features.py  linear_probe.py  __pycache__	  summarize.py
usage: finetune.py [-h] [--model MODEL] [--epochs EPOCHS] [--bs BS] [--lr LR]
                   [--head_lr HEAD_LR] [--wd WD] [--smoothing SMOOTHING]
                   [--img IMG] [--fold FOLD] [--workers WORKERS]
                   [--limit LIMIT] [--name NAME] [--tta]
                   [--eval_ckpt EVAL_CKPT]

options:
  -h, --help            show this help message and exit
  --model MODEL
  --epochs EPOCHS       passes over the training set
  --bs BS               images per training step
  --lr LR               learning rate of the pretrained backbone
  --head_lr HEAD_LR     learning rate of the new 67-class layer
  --wd WD               weight decay: regularization, plays the role 1/C did
  --smoothing SMOOTHING
                        label smoothing
  --img IMG             input image size
  --fold FOLD           validation fold; -1 = 

In [10]:
!python src/finetune.py \
    --model convnext_base.fb_in22k_ft_in1k \
    --epochs 10 \
    --bs 16 \
    --img 288 \
    --lr 1e-4 \
    --head_lr 5e-4 \
    --wd 0.05 \
    --smoothing 0.1 \
    --fold 0 \
    --name kevin_22k_headlr5e4


model.safetensors: downloading bytes:   3% 8.97M/354M [00:02<01:04, 5.35MB/s, 31.9kB/s  ]
model.safetensors: downloading bytes:  42% 147M/354M [00:03<00:01, 135MB/s, 11.6MB/s  ]
model.safetensors: downloading bytes:  49% 175M/354M [00:03<00:01, 136MB/s, 13.9MB/s  ]
model.safetensors: reconstructing file:   2% 6.91M/354M [00:03<02:35, 2.23MB/s,  434kB/s  ]
model.safetensors: downloading bytes:  55% 196M/354M [00:03<00:01, 125MB/s, 15.6MB/s  ]
model.safetensors: reconstructing file:   3% 11.3M/354M [00:03<01:03, 5.40MB/s,  697kB/s  ]
model.safetensors: downloading bytes:  63% 224M/354M [00:03<00:01, 129MB/s, 18.0MB/s  ]
model.safetensors: downloading bytes:  85% 300M/354M [00:04<00:00, 197MB/s, 23.5MB/s  ]
model.safetensors: downloading bytes:  91% 324M/354M [00:04<00:00, 206MB/s, 25.6MB/s  ]
model.safetensors: reconstructing file:  42% 149M/354M [00:04<00:01, 169MB/s, 7.95MB/s  ]  
model.safetensors: reconstructing file:  59% 209M/354M [00:04<00:00, 243MB/s, 13.2MB/s  ]
model.safetenso

In [11]:
import json

with open("results/runs/20261126-033641_Kevin Yang_kevin_22k_headlr5e4.json") as f:
    result = json.load(f)

print("Validation accuracy:", result["val_acc"])
print("vs_baseline:", result["val_acc"] - 0.771107)
print("Best epoch:", result["best_epoch"])

Validation accuracy: 0.8986866791744841
vs_baseline: 0.1275796791744841
Best epoch: 9


In [12]:
!python src/finetune.py \
    --model convnext_base.fb_in22k_ft_in1k \
    --epochs 10 \
    --bs 16 \
    --img 288 \
    --lr 1e-4 \
    --head_lr 2e-3 \
    --wd 0.05 \
    --smoothing 0.1 \
    --fold 0 \
    --name kevin_22k_headlr2e3

epoch 1/10  train_loss 1.993  val_acc 0.7927  (118s)
epoch 2/10  train_loss 1.107  val_acc 0.8527  (121s)
epoch 3/10  train_loss 0.884  val_acc 0.8668  (119s)
epoch 4/10  train_loss 0.818  val_acc 0.8809  (119s)
epoch 5/10  train_loss 0.784  val_acc 0.8931  (119s)
epoch 6/10  train_loss 0.763  val_acc 0.8902  (119s)
epoch 7/10  train_loss 0.755  val_acc 0.9015  (119s)
epoch 8/10  train_loss 0.752  val_acc 0.8996  (119s)
epoch 9/10  train_loss 0.750  val_acc 0.9053  (121s)
epoch 10/10  train_loss 0.748  val_acc 0.9015  (119s)
best val_acc 0.9053 at epoch 9; weights -> /content/data/checkpoints/kevin_22k_headlr2e3.pt
saved results/runs/20261126-040447_Kevin Yang_kevin_22k_headlr2e3.json


In [13]:
!pip freeze > requirements_colab.txt

In [15]:
!mkdir -p /content/drive/MyDrive/DAML_Kevin

!cp -r /content/repo/results /content/drive/MyDrive/DAML_Kevin/

!cp -r /content/data/checkpoints /content/drive/MyDrive/DAML_Kevin/

!cp /content/repo/requirements_colab.txt /content/drive/MyDrive/DAML_Kevin/

In [16]:
import json

with open("results/runs/20261126-040447_Kevin Yang_kevin_22k_headlr2e3.json") as f:
    result = json.load(f)

print("Validation accuracy:", result["val_acc"])
print("vs_baseline:", result["val_acc"] - 0.771107)
print("Best epoch:", result["best_epoch"])

Validation accuracy: 0.9052532833020638
vs_baseline: 0.1341462833020638
Best epoch: 9


In [17]:
!python src/extract_features.py \
    --model convnext_small.fb_in1k \
    --bs 16

!python src/linear_probe.py \
    --model convnext_small.fb_in1k \
    --C 0.01 \
    --fold 0 \
    --name kevin_convnextsmall1k_baseline

!python src/summarize.py


model.safetensors: downloading bytes:   0% 57.2k/201M [00:01<1:52:34, 29.7kB/s]
model.safetensors: downloading bytes:  18% 36.6M/201M [00:02<00:04, 34.5MB/s, 2.35MB/s  ]
model.safetensors: downloading bytes:  28% 57.2M/201M [00:02<00:03, 38.1MB/s, 4.67MB/s  ]
model.safetensors: reconstructing file:  14% 27.7M/201M [00:02<00:12, 14.0MB/s,  879kB/s  ]
model.safetensors: downloading bytes:  33% 65.5M/201M [00:03<00:03, 38.8MB/s, 5.45MB/s  ]
model.safetensors: downloading bytes:  40% 80.6M/201M [00:03<00:02, 43.7MB/s, 6.56MB/s  ]
model.safetensors: downloading bytes:  50% 100M/201M [00:03<00:02, 45.7MB/s, 8.05MB/s  ] 
model.safetensors: downloading bytes:  56% 112M/201M [00:03<00:01, 48.9MB/s, 8.89MB/s  ]
model.safetensors: downloading bytes:  59% 119M/201M [00:04<00:01, 53.2MB/s, 9.43MB/s  ]
model.safetensors: downloading bytes:  68% 136M/201M [00:04<00:01, 54.9MB/s, 10.7MB/s  ]
model.safetensors: downloading bytes:  90% 181M/201M [00:04<00:00, 81.6MB/s, 13.6MB/s  ]
model.safetensors: re